In [12]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

In [13]:
class AgentState(TypedDict):
    input: str
    intent: str
    cleaned_input: str
    response: str
    status: str
    feedback: str
    iterations: int

load_dotenv(override=True)

def call_llm(prompt: str):
    
    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=os.environ["HF_TOKEN"],
    )

    completion = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct:novita",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )

    return completion.choices[0].message.content



In [14]:
def cleaner(state: AgentState)-> AgentState:
    state["iterations"] = 0
    cleaner_prompt = f"""
    You are a helpful assistant that rewrites user questions clearly.

    Your task:
    - Convert the given question into a clear, complete, and grammatically correct question.
    - Keep the meaning exactly the same.
    - Do not add extra information.
    - Keep it concise.

    Return ONLY the rewritten question.

    User question:
    {state['input']}
    """

    cleaned_input = call_llm(cleaner_prompt)
    state["cleaned_input"] = cleaned_input
    return state
    

def router(state: AgentState)-> str:
    router_prompt = f"""
    You are an intent classification system.

    Classify input into ONLY ONE category:
    faq, technical, complaint

    Priority:
    complaint > technical > faq

    Rules:
    - Output ONLY one word
    - No explanation
    - No punctuation

    Examples:
    Input: My app keeps crashing
    Output: technical

    Input: What is refund policy
    Output: faq

    Input: I was charged twice and support is not responding
    Output: complaint

    User input:
    {state['cleaned_input']}
    """
    deciding_router = call_llm(router_prompt)
    if deciding_router.lower() == 'faq':
        return 'faq'
    elif deciding_router.lower() == 'technical':
        return 'technical'
    elif deciding_router.lower() == 'complaint':
        return 'complaint'
    else: 
        return END
    

def faq(state: AgentState)-> AgentState:
    state["intent"] = "faq"

    faq_prompt = f"""
    You are an informational customer support assistant.

    Your task:
    - Provide factual and concise answers
    - Keep tone neutral
    - Keep answer short

    Improve response using feedback if available.

    Feedback:
    {state.get('feedback', '')}

    User Question:
    {state['cleaned_input']}
    """

    final_response = call_llm(faq_prompt)
    state['response'] = final_response
    return state

def technical(state: AgentState)-> AgentState:
    state["intent"] = "technical"

    technical_prompt = f"""
    You are a technical support assistant.

    Your task:
    - Diagnose the issue
    - Give practical solutions
    - Provide step-by-step guidance

    Improve response using feedback if available.

    Feedback:
    {state.get('feedback', '')}

    User Issue:
    {state['cleaned_input']}
    """
    final_response = call_llm(technical_prompt)
    state['response'] = final_response
    return state

def complaint(state: AgentState)-> AgentState:
    state["intent"] = "complaint"

    complaint_prompt = f"""
    You are a customer support complaint handler.

    Your task:
    - Show empathy
    - Apologize briefly
    - Provide next steps
    - Do not invent company policies

    Improve response using feedback if available.

    Feedback:
    {state.get('feedback', '')}


    User Complaint:
    {state['cleaned_input']}
    """

    final_response = call_llm(complaint_prompt)
    state['response'] = final_response
    return state


def critic(state: AgentState)-> AgentState:
    critic_prompt = f"""
    You are a strict quality evaluator for a customer support AI system.

    Your job is to evaluate the given response based on the user query and intent.

    Intent: {state['intent']}

    Evaluation criteria:
    - Correctness: Does the response correctly address the user’s question?
    - Clarity: Is the response easy to understand?
    - Completeness: Does it provide enough useful information or next steps?
    - Tone: Is the tone appropriate for the intent?
      - FAQ → neutral and informative
      - Technical → helpful and solution-oriented
      - Complaint → empathetic and polite

    Rules:
    - If the response is good enough to be sent to the user → return status = "good"
    - If the response is missing important details or unclear → return status = "improve"
    - Be strict but fair (do not always return "improve")

    Output ONLY in JSON format:

    {{
      "status": "good" or "improve",
      "feedback": "short and specific reason if improvement is needed"
    }}

    User question:
    {state['cleaned_input']}

    Generated response:
    {state['response']}
    """

    critic_response = call_llm(critic_prompt)
    parsed_response = json.loads(critic_response)

    state['status'] = parsed_response['status']
    state['feedback'] = parsed_response['feedback']
    return state

    
def decision(state: AgentState):
    state['iterations'] += 1
    if state['status'].lower() == 'good':
        return END
    
    if state['iterations'] >= 2:
        return END
    
    return state['intent']

        

In [21]:
graph = StateGraph(AgentState)
graph.add_node('cleaner', cleaner)
graph.add_node('router', lambda state: state)
graph.add_node('faq', faq)
graph.add_node('technical', technical)
graph.add_node('complaint', complaint)
graph.add_node('critic', critic)



graph.add_edge(START, 'cleaner')
graph.add_edge('cleaner', 'router')
graph.add_conditional_edges(
    'router',
    router,
    {
        'faq': 'faq',
        'technical': 'technical',
        'complaint': 'complaint'
    }
)
graph.add_edge('faq', "critic")
graph.add_edge('technical', "critic")
graph.add_edge('complaint', "critic")

graph.add_conditional_edges(
    'critic',
    decision,
    {
        'faq': 'faq',
        'technical': 'technical',
        'complaint': 'complaint',
        END: END
    }
)

app = graph.compile()

In [ ]:
# answer = app.invoke({'input': "Can I cancel my order after placing it?"})
# print(answer['response'])